In [ ]:
import pandas as pd
#from openai import OpenAI
import torch
import numpy as np
import os
import matplotlib.pyplot as plt
import time, datetime
from sklearn.metrics.pairwise import cosine_similarity
import scipy.stats as stats
import ast
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from concurrent.futures import ThreadPoolExecutor

# Numerical Experiments

In [ ]:
#revised version 2
def compute_confidence_interval(results):
    # Convert results to a numpy array for statistical calculations, if not already
    results = np.array(results)
    
    # Calculate mean and standard deviation
    mean = np.mean(results)
    std_dev = np.std(results, ddof=1)  # ddof=1 for sample standard deviation
    std_error = std_dev / np.sqrt(len(results))
    z_score = stats.norm.ppf(0.975)  # two-tailed test, so 0.975 instead of 0.95
    margin_of_error = z_score * std_error

    confidence_interval = (mean - margin_of_error, mean + margin_of_error)

    return confidence_interval


def get_UCB(num_trials, num_positive, z_value, t, ctr_llm, n_llm):
    
    mean_val = num_positive / num_trials
    ucb = mean_val + z_value * np.sqrt(np.log(t + 2) / num_trials)

    ucb_llm = np.where(n_llm > 0,
                       (ctr_llm * n_llm + num_positive) / (num_trials + n_llm) + z_value * np.sqrt(np.log(t + 2) / (num_trials + n_llm)),
                       ucb)  # Default to ucb if n_llm <= 0
    
    return np.minimum(ucb, ucb_llm)


def simulate_bandit(df, T, n_value, z_value, ini = 1000):
    rng = np.random.RandomState(42)
    
    results = []
    results_ab = []

    sampled_ids = df['test_id'].drop_duplicates().sample(n=n_value, random_state=rng)
    sampled_df = df[df['test_id'].isin(sampled_ids)]
    
    
    for test_id, group in sampled_df.groupby('test_id'):
        # Initialize data structures for arms
        trials = np.ones(len(group))  # Start each arm with one trial initially
        successes = np.zeros(len(group))  # Initial clicks as successes
        ctr_llm = np.array(group['ini_CTR'].values)  # Initial CTR estimates
        #n_llm = np.array(group['initial_trials'].values)  # Initial number of trials
        n_llm = ini*np.ones(len(group))
        total_T = int(len(group)*T)

        
        rewards = np.zeros(total_T)
        
        for t in range(total_T):
            # Calculate UCB for each arm
            ucbs = get_UCB(trials, successes, z_value, t, ctr_llm, n_llm)
            
            # Choose the arm with the highest UCB
            chosen_arm = np.argmax(ucbs)
            ctr = group.iloc[chosen_arm]['CTR']
            
            # Simulate reward using the binomial distribution
            reward = rng.binomial(1, ctr)
            
            # Update trials and successes
            trials[chosen_arm] += 1
            successes[chosen_arm] += reward
            
            # Update cumulative rewards for this simulation
            rewards[t] = reward
        
        # Sum of rewards for this test_id
        results.append(np.sum(rewards))
        
    return results        


def get_Thompson_Sampling(alpha, beta, n_llm, ctr_llm):
    # Sample from the Beta distribution
    samples = np.random.beta(alpha + ctr_llm * n_llm, beta + (1 - ctr_llm) * n_llm)
    return samples

def simulate_thompson_sampling(df, T, n_value, a_ = 1, b_ = 100, ini = 1000):
    # LOLA: LLM+Thompson Sampling if n_value > 0
    # Pure Thompson Sampling: if n_value == 0
    rng = np.random.RandomState(42)
    
    results = []
    
    sampled_ids = df['test_id'].drop_duplicates().sample(n=n_value, random_state=rng)
    sampled_df = df[df['test_id'].isin(sampled_ids)]
    
    for test_id, group in sampled_df.groupby('test_id'):
        # Initialize data structures for arms
        trials = np.ones(len(group))  # Start each arm with one trial initially
        successes = np.zeros(len(group))  # Initial clicks as successes
        alpha = a_*np.ones(len(group))  # Prior alpha (successes)
        beta = b_*np.ones(len(group))   # Prior beta (failures)
        ctr_llm = np.array(group['ini_CTR'].values)  # Initial CTR estimates
        ctr_llm = np.maximum(ctr_llm, 0)
        #n_llm = np.array(group['initial_trials'].values)  # Initial number of trials
        n_llm = ini*np.ones(len(group))
        total_T = int(len(group) * T)
        
        rewards = np.zeros(total_T)
        
        for t in range(total_T):
            # Sample from the Beta distribution for each arm
            ts_samples = get_Thompson_Sampling(alpha, beta, n_llm, ctr_llm)
            
            # Choose the arm with the highest sample value
            chosen_arm = np.argmax(ts_samples)
            ctr = group.iloc[chosen_arm]['CTR']
            
            # Simulate reward using the binomial distribution
            reward = rng.binomial(1, ctr)
            
            # Update trials and successes
            trials[chosen_arm] += 1
            successes[chosen_arm] += reward
            
            # Update alpha and beta parameters
            alpha[chosen_arm] += reward
            beta[chosen_arm] += (1 - reward)
            
            # Update cumulative rewards for this simulation
            rewards[t] = reward
        
        # Sum of rewards for this test_id
        results.append(np.sum(rewards))
        
    return results

def simulate_LLM_no_bandit(df, T, n_value):
    rng = np.random.RandomState(42)
    
    results_no_bandit = []

    sampled_ids = df['test_id'].drop_duplicates().sample(n=n_value, random_state=rng)
    sampled_df = df[df['test_id'].isin(sampled_ids)]
    
    
    for test_id, group in sampled_df.groupby('test_id'):
        total_T = int(len(group)*T)
        rewards_no_bandit = np.zeros(total_T)
        #for t in range(total_T):
        best_arm = np.argmax(group['ini_CTR'])
        ctr = group.iloc[best_arm]['CTR']
        reward = rng.binomial(1, ctr, total_T)
        rewards_no_bandit = reward

        results_no_bandit.append(np.sum(rewards_no_bandit))
        
    return results_no_bandit

def simulate_ab(df, T, n_value, ratio = 0.2):
    #T is the traffic per headline
    rng = np.random.RandomState(42)

    results_ab = []

    sampled_ids = df['test_id'].drop_duplicates().sample(n=n_value, random_state=rng)
    sampled_df = df[df['test_id'].isin(sampled_ids)]
    
    
    for test_id, group in sampled_df.groupby('test_id'):
        total_T = int(len(group)*T)
        
        # Initialize data structures for arms
        trials = np.ones(len(group))  # Start each arm with one trial initially
        successes = np.zeros(len(group))  # Initial clicks as successes
        explore_T = int(ratio * total_T)  # 20% of traffic for exploration/test
        exploit_T = total_T - explore_T  # 80% of traffic for exploitation/roll
        rewards_ab = np.zeros(total_T)
        # Exploration phase
        chosen_arm = rng.choice(len(group), explore_T)
        ctr = group.iloc[chosen_arm]['CTR']
        reward = rng.binomial(1, p = ctr)
        np.add.at(trials, chosen_arm, 1)
        np.add.at(successes, chosen_arm, reward)
        np.add.at(rewards_ab, np.arange(explore_T), reward)
        
        empirical_ctrs = successes / trials
        best_arm = np.argmax(empirical_ctrs)
        ctr = group.iloc[best_arm]['CTR']
        reward = rng.binomial(1, ctr, total_T-explore_T)
        rewards_ab[explore_T:total_T] = reward
        results_ab.append(np.sum(rewards_ab))
    return results_ab



## Data Process and Embedding-based CTR Prediction

In [ ]:
DIM = 3072
print('get data from csv')
df = pd.read_csv('all_test_headline_embed_3072.csv')


df = df[['new_test_id', 'headline', 'embedding_headline', 'impressions', 'clicks']]
df.loc[:, 'CTR'] = df['clicks'] / df['impressions']
df = df.rename(columns={
    'new_test_id': 'test_id',
    'embedding_headline': 'embedding'
})


embeddings_df = df['embedding'].str.split(pat=", ", expand=True)
embeddings_df.columns = [f'emb_{i}' for i in range(embeddings_df.shape[1])]
embeddings_df['emb_3071'] = embeddings_df['emb_3071'].str.rstrip(']')
embeddings_df['emb_0'] = embeddings_df['emb_0'].apply(
    lambda x: x[1:] if isinstance(x, str) else x
)

In [ ]:
df

In [ ]:
embeddings_df = embeddings_df.astype(float)
embeddings_df.astype(float)
df = pd.concat([df, embeddings_df], axis=1)
df = df.drop('embedding', axis=1)
del embeddings_df
print(len(df))

In [ ]:
train_ratio = 0.7
calibrate_ratio = 0.1
test_ratio = 0.2


gss = GroupShuffleSplit(n_splits=1, test_size=test_ratio+calibrate_ratio, random_state=42)
train_idx, temp_idx = next(gss.split(df, groups=df['test_id']))
train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=test_ratio/(test_ratio+calibrate_ratio), random_state=42)
calibrate_idx, test_idx = next(gss2.split(temp_df, groups=temp_df['test_id']))
calibrate_df = temp_df.iloc[calibrate_idx]
test_df = temp_df.iloc[test_idx]
del temp_df
print(len(test_df))
print(len(calibrate_df))

In [ ]:
unique_headlines_train = set(train_df['headline'].unique())
test_df = test_df[~test_df['headline'].isin(unique_headlines_train)]
calibrate_df = calibrate_df[~calibrate_df['headline'].isin(unique_headlines_train)]
print(len(test_df))
print(len(calibrate_df))

In [ ]:
#generate dataset for LoRA fine-tuning Llama-3-8b or other prediction models
#test_df[['test_id','headline','impressions','clicks','CTR']].to_csv('LoRA_CTR_test.csv')
#train_df[['test_id','headline','impressions','clicks','CTR']].to_csv('LoRA_CTR_train.csv')

In [ ]:
import sys
sys.setrecursionlimit(10000)  # Increase the recursion limit; the default is usually 1000

formula = 'CTR ~ ' + ' + '.join(train_df.columns.drop(['CTR', 'test_id', 'headline', 'impressions', 'clicks']))
#mod = smf.quantreg(formula, train_df)
mod2 = smf.ols(formula, train_df)
#res_median = mod.fit(q=0.5)
res_mean = mod2.fit()

train_df.loc[:, 'ini_CTR'] = res_mean.predict(train_df)
calibrate_df.loc[:, 'ini_CTR'] = res_mean.predict(calibrate_df)
test_df.loc[:, 'ini_CTR'] = res_mean.predict(test_df)
train_df['ini_CTR'] = train_df['ini_CTR'].apply(lambda x: 0.01 if pd.isna(x) or x < 0 else x)
calibrate_df['ini_CTR'] = calibrate_df['ini_CTR'].apply(lambda x: 0.01 if pd.isna(x) or x < 0 else x)
test_df['ini_CTR'] = test_df['ini_CTR'].apply(lambda x: 0.01 if pd.isna(x) or x < 0 else x)


r2_test = r2_score(test_df['CTR'], test_df['ini_CTR'])
print(f"R2 Score using OpenAI embeddings: {r2_test}")

df_lora = pd.read_csv('LoRA CTR.csv')
r2_lora = r2_score(df_lora['CTR'], df_lora['ini_CTR'])
print(f"R2 Score using LoRA fine-tuned Llama-3-8b: {r2_lora}")

In [ ]:
r2_test = r2_score(test_df['CTR'], test_df['ini_CTR'])
print(f"R2 Score using OpenAI embeddings: {r2_test}")

df_lora = pd.read_csv('LoRA CTR.csv')
r2_lora = r2_score(df_lora['CTR'], df_lora['ini_CTR'])
print(f"R2 Score using LoRA fine-tuned Llama-3-8b: {r2_lora}")

r2_mean = r2_score(test_df['CTR'], train_df['CTR'].mean()*np.ones(len(test_df)))
print(f"R2 Score if using mean CTR of training data: {r2_mean}")

In [ ]:
from itertools import combinations

print('Performance of Embeddings in test dataset')

correct_count = 0
grouped = test_df.groupby('test_id')
for name, group in grouped:
    max_ctr_idx = group['CTR'].idxmax()
    max_ini_ctr_idx = group['ini_CTR'].idxmax()
    if max_ctr_idx == max_ini_ctr_idx:
        correct_count += 1
total_groups = len(grouped)
accuracy = correct_count / total_groups

print(f'Accuracy using embedding CTR in test data: {accuracy:.4f}')

total_accuracy = 0
grouped = test_df.groupby('test_id')
for name, group in grouped:
    num_items = len(group)
    total_accuracy += 1 / num_items
average_accuracy = total_accuracy / len(grouped)

print(f'Accuracy via Random Guess in test data: {average_accuracy:.4f}')


correct_pairs = 0
total_pairs = 0
grouped = test_df.groupby('test_id')
for name, group in grouped:
    items = list(group.index)
    pairs = list(combinations(items, 2))
    for (i, j) in pairs:
        total_pairs += 1
        if (group.loc[i, 'ini_CTR'] > group.loc[j, 'ini_CTR']) == (group.loc[i, 'CTR'] > group.loc[j, 'CTR']):
            correct_pairs += 1
accuracy = correct_pairs / total_pairs
print(f'Accuracy in paired headlines: {accuracy:.4f}')

print('------------------------------------')
print('Performance of Embeddings in test dataset')
correct_count = 0
grouped = train_df.groupby('test_id')
for name, group in grouped:
    max_ctr_idx = group['CTR'].idxmax()
    max_ini_ctr_idx = group['ini_CTR'].idxmax()
    if max_ctr_idx == max_ini_ctr_idx:
        correct_count += 1
total_groups = len(grouped)
accuracy = correct_count / total_groups
print(f'Accuracy using embedding CTR in training data: {accuracy:.4f}')

total_accuracy = 0
grouped = train_df.groupby('test_id')
for name, group in grouped:
    num_items = len(group)
    total_accuracy += 1 / num_items
average_accuracy = total_accuracy / len(grouped)

print(f'Accuracy via Random Guess in training data: {average_accuracy:.4f}')

correct_pairs = 0
total_pairs = 0
grouped = test_df.groupby('test_id')
for name, group in grouped:
    items = list(group.index)
    pairs = list(combinations(items, 2))
    for (i, j) in pairs:
        total_pairs += 1
        if (group.loc[i, 'ini_CTR'] > group.loc[j, 'ini_CTR']) == (group.loc[i, 'CTR'] > group.loc[j, 'CTR']):
            correct_pairs += 1
accuracy = correct_pairs / total_pairs
print(f'Accuracy in paired headlines: {accuracy:.4f}')

In [ ]:
y_test = calibrate_df['CTR']
y_test_pred = calibrate_df['ini_CTR']

plt.scatter(y_test, y_test_pred, s=10, alpha=0.5)  # s controls the size of the dots

# Plot the y=x line (diagonal line)
max_value = max(y_test.max(), y_test_pred.max())  # Get the maximum value for x and y axis limits
min_value = min(y_test.min(), y_test_pred.min())  # Get the minimum value for x and y axis limits
plt.plot([min_value, max_value], [min_value, max_value], 'k--', lw=2)  # 'k--' means black dashed line, lw is line width

# Labeling the axes
plt.xlabel('True CTR')
plt.ylabel('Predicted CTR')

# Show the plot with a grid
plt.grid(True)
plt.show()

percentile_90 = np.percentile(np.hstack((y_test, y_test_pred)), 90)

# Filter out the top 10% of outliers
mask = y_test <= percentile_90
y_test_filtered = y_test[mask]
y_test_pred_filtered = y_test_pred[mask]

plt.figure(figsize=(10, 10))
plt.scatter(y_test_filtered, y_test_pred_filtered, s=5, alpha=0.5)
plt.plot([0, percentile_90], [0, percentile_90], 'k--', lw=2)

# Labeling the axes
plt.xlabel('True CTR')
plt.ylabel('Predicted CTR')

# Set the axes to be equal
plt.axis('square')

# Set the same scale for both axes
plt.xlim(0, percentile_90)
plt.ylim(0, percentile_90)

# Show the plot with a grid
plt.grid(True)
plt.show()


In [ ]:
def calculate_beta_parameters(mean, variance):

    # Calculate alpha using the derived formula
    alpha = mean * ((mean * (1 - mean) / variance) - 1)
    
    # Calculate beta using the derived formula
    beta = (1 - mean) * ((mean * (1 - mean) / variance) - 1)
    
    return alpha, beta


alpha_train, beta_train = calculate_beta_parameters(train_df['CTR'].mean(), train_df['CTR'].var())
print("Alpha:", alpha_train)
print("Beta:", beta_train)

In [ ]:
del train_df, df

## Fine-tune algorithms
One can skip this cell because this procedure is time consuming and we have fine-tuned all algorithms' hyperparameters from our side.

In [ ]:
print('----------------------------------')
print('Fine tuning hyperparameters in algorithms')
n_calibrate = calibrate_df.groupby('test_id').ngroups


#Taking fine-tuning LLM-2UCBs for example, UCB and TS follow the same procedure.
T = 600#take T=600 for example
initial_trials_range = [600, 800, 1000, 1200, 1400]
z_values_range = [0.02, 0.04, 0.06, 0.08, 0.10]

results = []

for initial_trials in initial_trials_range:
    for z_ in z_values_range:
        calibrate_df['initial_trials'] = initial_trials
        ucb2_result = simulate_bandit(calibrate_df, T, n_calibrate, z_, ini=initial_trials)
        results.append((initial_trials, z_, ucb2_result))

results_df = pd.DataFrame(results, columns=['initial_trials', 'z_', 'ucb2_result'])

results_df['performance'] = results_df.apply(
    lambda row: np.mean(row['ucb2_result']), axis=1
)

heatmap_data = results_df.pivot(index="initial_trials", columns="z_", values="performance")

# Plotting the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="viridis")
plt.title("Average Click of LLM-2UCBs")
plt.ylabel("Initial Trials")
plt.xlabel("Z-value")
plt.show()



#fine-tune E&C
ratio_range = [0.1, 0.2, 0.3, 0.4, 0.5]
best_ratio = ratio_range[0]
best_r_ratio = -1
for ratio_ in ratio_range:
    ab_result = simulate_ab(calibrate_df, T, n_calibrate, ratio_)
    if np.mean(ab_result) > best_r_ratio:
        best_r_ratio = np.mean(ab_result)
        best_ratio = ratio_
print('best ratio = ',  best_ratio)

alpha_train = 1.3833788217740643
beta_train = 96.11247058958283

## Run LOLA and Benchmarks on test data

In [ ]:
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
import os
from multiprocessing import Manager
from threading import Lock
import time


# Setup a manager for multiprocessing that can handle locks
manager = Manager()
lock = manager.Lock()

#choose LoRA fine-tuned Llama-3-8b for CTR prediction
test_df = pd.read_csv('LoRA CTR.csv')


def run_simulation(T):
    
    #define all hyperparameters in algorithms
    best_z_ucb1 = 0.08
    best_z_ucb2 = 0.08
    fixed_n_llm = 1000
    fixed_n_llm_ts = 1200
    best_ratio = 0.2#ratio in E&C for exploration
    
    n_test = test_df.groupby('test_id').ngroups
    
    print('test for pure LLM at T = ', T)
    no_bandit_result = simulate_LLM_no_bandit(test_df, T, n_test)
    
    print('test for EC at T = ', T)
    ab_result = simulate_ab(test_df, T, n_test, best_ratio)
    
    print('test for UCBs at T = ', T)
    ucb1_result = simulate_bandit(test_df, T, n_test, best_z_ucb1, ini=0)
    ucb2_result = simulate_bandit(test_df, T, n_test, best_z_ucb2, ini=fixed_n_llm)

    print('test for TS at T = ', T)
    ts_result = simulate_thompson_sampling(test_df, T, n_test, a_=alpha_train, b_=beta_train, ini=0)
    ts_llm_result = simulate_thompson_sampling(test_df, T, n_test, a_=alpha_train, b_=beta_train, ini=fixed_n_llm_ts)

    ucb1_result = np.array(ucb1_result)
    ucb2_result = np.array(ucb2_result)
    no_bandit_result = np.array(no_bandit_result)
    ab_result = np.array(ab_result)
    ts_result = np.array(ts_result)
    ts_llm_result = np.array(ts_llm_result)

    result = (T, best_z_ucb1, best_z_ucb2, best_ratio, ab_result, no_bandit_result, ucb1_result, ucb2_result, ts_result, ts_llm_result)
    save_results(result)
    print('Finished T = ', T)
    print('----------------------------------')


def save_results(result):
    lock.acquire()
    try:
        file_exists = os.path.exists('simulation_results_regret_min.csv')
        with open('simulation_results_regret_min.csv', 'a') as f:
            result_simulation_df = pd.DataFrame([result], columns=[
                'T', 'z_ucb1', 'z_ucb2', 'best_ratio', 'ab_result', 'no_bandit_result',
                'ucb1_result', 'ucb2_result', 'ts_result', 'ts_llm_result'
            ])
            result_simulation_df.to_csv(f, header=not file_exists, index=False)
    except Exception as e:
        print(f"Failed to write results: {e}")
    finally:
        lock.release()

def main():
    T_list = [50, 100, 200, 400, 600, 800, 1000]
    with ProcessPoolExecutor(max_workers = 3) as executor:
        executor.map(run_simulation, T_list)


In [ ]:
#main() #run in parallel
T_list = [50, 100, 200, 400, 600, 800, 1000]
for T in T_list:
    run_simulation(T)

# Visualize numerical results 

In [ ]:
result_simulation_df = pd.read_csv('simulation_results_regret_min.csv')

plt.figure(figsize=(10, 6))
result_simulation_df['T'] = pd.to_numeric(result_simulation_df['T'], errors='coerce')

markers = ['o', 's', 'D', '^']
labels = ['LLM-2UCBs (LOLA)', 'UCB', 'Pure LLM', 'E&C']
columns = ['ucb2', 'ucb1', 'no_bandit', 'ab']

linestyles = ['-', '--', '-.', ':']
colors = ['blue', 'green', 'red', 'purple']


for text_, label, marker, linestyle, color in zip(columns, labels, markers, linestyles, colors):
    result_simulation_df[text_+'_result'] = result_simulation_df[text_+'_result'].apply(ast.literal_eval)
    #result_simulation_df[text_+'_result'] = pd.to_numeric(result_simulation_df[text_+'_result'], errors='coerce')

    #result_simulation_df[text_+'_result'] = np.array(result_simulation_df[text_+'_result'])
    result_simulation_df[text_ + '_mean'] = result_simulation_df.apply(
        lambda row: np.mean(np.array([row[text_ + '_result']]) / row['T']), axis=1)

    #result_simulation_df[text_ + '_lb'] = result_simulation_df.apply(
    #    lambda row: compute_confidence_interval(np.array([row[text_ + '_result']]) / row['T'])[0], axis=1)

    #result_simulation_df[text_ + '_ub'] = result_simulation_df.apply(
    #    lambda row: compute_confidence_interval(np.array([row[text_ + '_result']]) / row['T'])[1], axis=1)

    plt.plot(result_simulation_df['T'], result_simulation_df[text_ + '_mean'], marker=marker, linestyle=linestyle, color=color, label=label)


plt.xlabel(r'Traffic/Impressions per headline in tests $\tau$')
plt.ylabel('Average clicks per test per period')
plt.xticks([50, 100, 200, 400, 600, 800, 1000])
#plt.xticks([50, 100, 200])

plt.xlim([50, 1000])
plt.legend()
plt.grid(True)
plt.savefig('2ucb_simulation_compr.pdf', format='pdf', bbox_inches='tight')
plt.show()


result_simulation_df = pd.read_csv('simulation_results_regret_min.csv')

plt.figure(figsize=(10, 6))
result_simulation_df['T'] = pd.to_numeric(result_simulation_df['T'], errors='coerce')

markers = ['o', 's', 'D', '^', '<', 'v']
labels = ['LLM-2UCBs (LOLA)', 'UCB', 'Pure LLM', 'E&C', 'LLM-TS (LOLA)', 'TS']
columns = ['ucb2', 'ucb1', 'no_bandit', 'ab', 'ts_llm', 'ts']

linestyles = ['-', '--', '-.', ':', '-', '-']
colors = ['blue', 'green', 'red', 'purple', 'brown', 'black']


for text_, label, marker, linestyle, color in zip(columns, labels, markers, linestyles, colors):
    result_simulation_df[text_+'_result'] = result_simulation_df[text_+'_result'].apply(ast.literal_eval)
    #result_simulation_df[text_+'_result'] = pd.to_numeric(result_simulation_df[text_+'_result'], errors='coerce')

    #result_simulation_df[text_+'_result'] = np.array(result_simulation_df[text_+'_result'])
    result_simulation_df[text_ + '_mean'] = result_simulation_df.apply(
        lambda row: np.mean(np.array([row[text_ + '_result']]) / row['T']), axis=1)

    #result_simulation_df[text_ + '_lb'] = result_simulation_df.apply(
    #    lambda row: compute_confidence_interval(np.array([row[text_ + '_result']]) / row['T'])[0], axis=1)

    #result_simulation_df[text_ + '_ub'] = result_simulation_df.apply(
    #    lambda row: compute_confidence_interval(np.array([row[text_ + '_result']]) / row['T'])[1], axis=1)

    plt.plot(result_simulation_df['T'], result_simulation_df[text_ + '_mean'], marker=marker, linestyle=linestyle, color=color, label=label)


plt.xlabel(r'Traffic/Impressions per headline in tests $\tau$')
plt.ylabel('Average clicks per test per period')
plt.xticks([50, 100, 200, 400, 600, 800, 1000])
#plt.xticks([50, 100, 200])
plt.xlim([50, 1000])
plt.legend()
plt.grid(True)
plt.savefig('2ucb_simulation_compr_ts.pdf', format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
import itertools
from scipy.stats import ttest_rel


# Map the labels to their respective columns in the dataframe
label_to_column = {
    'LLM-2UCBs': 'ucb2',
    'UCB': 'ucb1',
    'Pure LLM': 'no_bandit',
    'E&C': 'ab',
    'LLM-TS':'ts_llm',
    'TS':'ts'
}
comparisons = list(itertools.combinations(label_to_column.keys(), 2))


# Prepare the result dataframe
comparison_results = pd.DataFrame({
    'T': result_simulation_df['T']
})

# Compute the mean differences and p-values for each comparison
for label1, label2 in comparisons:
    col1 = label_to_column[label1]
    col2 = label_to_column[label2]
    
    mean_diff_col = f'{label1} v.s. {label2}_mean_diff'
    p_value_col = f'{label1} v.s. {label2}_p_value'
    
    mean_diffs = []
    p_values = []
    
    for _, row in result_simulation_df.iterrows():
        data1 = np.array([row[f'{col1}_result']]) / row['T']
        data2 = np.array([row[f'{col2}_result']]) / row['T']
        
        mean_diff = np.mean(data1[0] - data2[0])/np.mean(data2[0])*100
        t_stat, p_value = ttest_rel(data1[0], data2[0])
        
        mean_diffs.append(mean_diff)
        p_values.append(p_value)
    
    comparison_results[mean_diff_col] = mean_diffs
    comparison_results[p_value_col] = p_values



In [ ]:
comparison_results

# Appendix - Get embeddings

In [ ]:
#We provide the code to get embeddings from OpenAI, as well as the returned embedding stored in CSV.

from openai import OpenAI
import concurrent.futures
import time
from threading import Lock
import os

client = OpenAI(api_key='XXX')

DIM = 3072
MODEL = "text-embedding-3-large"


# Assuming the DataFrame and other necessary variables are defined
# ...

class RateLimiter:
    def __init__(self, max_requests, per_seconds):
        self.max_requests = max_requests
        self.per_seconds = per_seconds
        self.lock = Lock()
        self.start_time = time.time()
        self.request_count = 0

    def wait(self):
        with self.lock:
            self.request_count += 1
            if self.request_count >= self.max_requests:
                elapsed = time.time() - self.start_time
                if elapsed < self.per_seconds:
                    time.sleep(self.per_seconds - elapsed)
                self.start_time = time.time()
                self.request_count = 0

# Initialize the rate limiter for 2000 requests per minute
rate_limiter = RateLimiter(2000, 60)

def get_embedding(text):
    rate_limiter.wait()
    try:
        response = client.embeddings.create(input=[text], model=MODEL, dimensions=DIM)
        return response.data[0].embedding
    except Exception as e:
        print(f"Error fetching embedding for text: {text}. Error: {e}")
        return None  # Return None if error

def add_embeddings(df, column_name):
    embeddings = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        results = executor.map(get_embedding, df[column_name])
        embeddings = list(results)

    df[f'embedding_{column_name.split("_")[-1]}'] = embeddings

    
    
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
file_path = os.path.join(parent_dir, 'upworthy-archive-datasets/ctr-all.csv')
df = pd.read_csv(file_path)
df = df[['clickability_test_id', 'eyecatcher_id', 'headline','impressions','clicks']]
df['new_test_id'] = df.groupby(['clickability_test_id', 'eyecatcher_id']).ngroup() + 1
df['clicks'] = df['clicks'].astype(float) 
df['impressions'] = df['impressions'].astype(float) 
df['CTR'] = df['clicks']/df['impressions']
    
print('getting embeddings for headline')
add_embeddings(df, 'headline')

print('save embeddings')
df.to_csv('all_test_headline_embed_3072.csv')